In [ ]:
# NeuralSpace Python environment includes common analytics libraries
# You can run this cell with Shift+Enter

import numpy as np  # linear algebra
import pandas as pd  # data processing (e.g. pd.read_csv)

# Example: list files in your mounted input directory
import os
for dirname, _, filenames in os.walk("/workspace/input"):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write outputs to your working directory for this session

In [ ]:
# Dataset: Iris Dataset
# Dataset ID: ds_001
# Type: tabular | Size: 4.4 KB
# Mounted at (expected): /workspace/datasets/iris_dataset
DATASET_PATH = "/workspace/datasets/iris_dataset"

from pathlib import Path

_DATASET_CANDIDATES = [
    Path(DATASET_PATH),
    Path("/workspace/datasets/iris_dataset"),
    Path("/workspace/datasets/ds_001"),
    Path("/workspace/datasets/iris_dataset"),
    Path("/workspace/input/ds_001"),
]

DATASET_DIR = next((p for p in _DATASET_CANDIDATES if p.exists()), None)

if DATASET_DIR is None:
    datasets_root = Path("/workspace/datasets")
    if datasets_root.exists():
        subdirs = [d.name for d in datasets_root.iterdir() if d.is_dir()]
        print(f"[WARN] Dataset 'ds_001' is not mounted in this workspace. Available: {subdirs}")
    else:
        print("[WARN] /workspace/datasets does not exist in this runtime.")
else:
    print(f"Resolved dataset path: {DATASET_DIR}")

import pandas as pd

if DATASET_DIR is None:
    print("[WARN] Dataset is not mounted in this workspace. Skipping tabular preview.")
else:
    csv_files = sorted(DATASET_DIR.rglob("*.csv"))
    print(f"Found {len(csv_files)} csv files")

    if csv_files:
        df = pd.read_csv(csv_files[0])
        print(f"Preview file: {csv_files[0]}")
        print(df.shape)
        display(df.head())
    else:
        print("[DEV] No CSV files found in resolved dataset path.")

In [ ]:
# Model: ResNet50 Smoke Test
# Architecture: resnet50 | Framework: onnx
# Task: image_classification | Size: 63.0 B
# Mounted at (expected): /workspace/models/resnet50_smoke_test
MODEL_PATH = "/workspace/models/resnet50_smoke_test"

from pathlib import Path

_MODEL_CANDIDATES = [
    Path(MODEL_PATH),
    Path("/workspace/models/resnet50_smoke_test"),
    Path("/workspace/models/resnet50_smoke_test"),
    Path("/workspace/models"),
]

MODEL_DIR = next((p for p in _MODEL_CANDIDATES if p.exists() and p.is_dir() and any(p.rglob("*"))), None)

if MODEL_DIR is None:
    models_root = Path("/workspace/models")
    if models_root.exists():
        subdirs = [d.name for d in models_root.iterdir() if d.is_dir()]
        print(f"[WARN] Model 'ResNet50 Smoke Test' is not mounted in this workspace. Available dirs: {subdirs}")
    else:
        print("[WARN] /workspace/models does not exist in this runtime.")
else:
    print(f"Resolved model path: {MODEL_DIR}")

import onnxruntime as ort

if MODEL_DIR is None:
    print("[WARN] Model path not found. Skipping ONNX load test.")
else:
    onnx_files = sorted(MODEL_DIR.rglob("*.onnx"))

    if not onnx_files:
        print("[WARN] No .onnx file found under resolved model path.")
    else:
        selected_model = None

        # Validate candidates first to avoid noisy stack traces from corrupted files.
        for candidate in onnx_files:
            try:
                session = ort.InferenceSession(str(candidate))
                selected_model = str(candidate)
                print(f"[OK] Loaded ONNX: {selected_model}")
                print("Providers:", session.get_providers())
                break
            except Exception as e:
                print(f"[WARN] Skip invalid ONNX: {candidate.name} -> {type(e).__name__}")

        if selected_model is None:
            print("[WARN] All discovered .onnx files are invalid/corrupted.")
            print("[INFO] Creating a minimal smoke-test ONNX model so you can continue system testing.")

            try:
                import onnx
                from onnx import helper, TensorProto
                import numpy as np

                smoke_path = MODEL_DIR / "smoke_test.onnx"

                x = helper.make_tensor_value_info("x", TensorProto.FLOAT, [1, 3])
                y = helper.make_tensor_value_info("y", TensorProto.FLOAT, [1, 3])
                one = helper.make_tensor("one", TensorProto.FLOAT, [1, 3], [1.0, 1.0, 1.0])
                add = helper.make_node("Add", inputs=["x", "one"], outputs=["y"])

                graph = helper.make_graph([add], "smoke_graph", [x], [y], [one])
                model = helper.make_model(graph, producer_name="workspace_smoke_test")
                onnx.save(model, str(smoke_path))

                session = ort.InferenceSession(str(smoke_path))
                output = session.run(None, {"x": np.array([[2.0, 3.0, 4.0]], dtype=np.float32)})

                print(f"[OK] Smoke model created: {smoke_path}")
                print("[OK] Smoke inference output:", output[0])
            except Exception as e:
                print("[ERROR] Could not create fallback smoke ONNX model.")
                print(f"[ERROR] {type(e).__name__}: {e}")

Resolved model path: /workspace/models
[OK] Loaded ONNX: /workspace/models/model_ee2d8dd13626_sample-model.onnx
Providers: ['CPUExecutionProvider']
